In [36]:

import torch
import torchvision
import numpy as np
import pandas as pd
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GB"
    )


PyTorch: 2.14.1+cu126
Torchvision: 0.29.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory: 8.0 GB


In [37]:

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# Select device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained MobileNetV3-Large
spatial_backbone = mobilenet_v3_large(
    weights=MobileNet_V3_Large_Weights.DEFAULT
)

# Remove original classification head
spatial_backbone.classifier = nn.Identity()

spatial_backbone = spatial_backbone.to(DEVICE)
spatial_backbone.eval()

print("Spatial backbone loaded successfully!")
print("Architecture: MobileNetV3-Large")
print("Device:", DEVICE)


Spatial backbone loaded successfully!
Architecture: MobileNetV3-Large
Device: cuda


In [38]:

# Test with one batch of 160x160 RGB images

dummy_images = torch.randn(2, 3, 160, 160).to(DEVICE)

with torch.no_grad():
    spatial_features = spatial_backbone(dummy_images)

print("Input shape:", dummy_images.shape)
print("Spatial feature shape:", spatial_features.shape)
print("Feature dimension:", spatial_features.shape[1])
print("Output contains NaN:", torch.isnan(spatial_features).any().item())
print("Output contains Inf:", torch.isinf(spatial_features).any().item())


Input shape:

 torch.Size([2, 3, 160, 160])
Spatial feature shape: torch.Size([2, 960])
Feature dimension: 960
Output contains NaN: False
Output contains Inf: False


In [39]:

try:
    import pywt
    print("PyWavelets version:", pywt.__version__)
    print("Haar wavelet available:", "haar" in pywt.wavelist())
except ImportError:
    print("PyWavelets is not installed.")


PyWavelets version: 1.8.0
Haar wavelet available: True


In [40]:
%pip install PyWavelets

Note: you may need to restart the kernel to use updated packages.


In [41]:

import torch
import torch.nn as nn
import pywt
import numpy as np

class FrequencyFeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, images):

        # Convert RGB to grayscale
        gray = images.mean(dim=1, keepdim=True)

        # 1. FFT features
        fft = torch.fft.fft2(gray)
        fft_shift = torch.fft.fftshift(fft, dim=(-2, -1))

        fft_magnitude = torch.log1p(torch.abs(fft_shift))

        # 2. Haar wavelet features
        haar_features = []

        for image in gray:

            image_np = image.squeeze(0).detach().cpu().numpy()

            coeffs = pywt.wavedec2(
                image_np,
                wavelet="haar",
                level=2
            )

            # Keep approximation coefficients
            wavelet_parts = [coeffs[0]]

            # Add horizontal, vertical and diagonal details
            for horizontal, vertical, diagonal in coeffs[1:]:

                wavelet_parts.append(horizontal)
                wavelet_parts.append(vertical)
                wavelet_parts.append(diagonal)

            # Flatten all coefficient arrays
            flattened = np.concatenate([
                np.asarray(part).reshape(-1)
                for part in wavelet_parts
            ])

            haar_features.append(flattened)

        haar_features = torch.tensor(
            np.stack(haar_features),
            dtype=images.dtype,
            device=images.device
        )

        return fft_magnitude, haar_features


# Initialize corrected extractor
frequency_extractor = FrequencyFeatureExtractor().to(DEVICE)

print("Corrected frequency extractor initialized!")


Corrected frequency extractor initialized!


In [42]:

# Test using a batch of 2 RGB images
dummy_images = torch.randn(2, 3, 160, 160).to(DEVICE)

fft_features, haar_features = frequency_extractor(dummy_images)

print("Input shape:", dummy_images.shape)
print("FFT feature shape:", fft_features.shape)
print("Haar feature shape:", haar_features.shape)

print("FFT contains NaN:", torch.isnan(fft_features).any().item())
print("FFT contains Inf:", torch.isinf(fft_features).any().item())

print("Haar contains NaN:", torch.isnan(haar_features).any().item())
print("Haar contains Inf:", torch.isinf(haar_features).any().item())


Input shape: torch.Size([2, 3, 160, 160])
FFT feature shape: torch.Size([2, 1, 160, 160])
Haar feature shape: torch.Size([2, 25600])
FFT contains NaN: False
FFT contains Inf: False
Haar contains NaN: False
Haar contains Inf: False


In [43]:

class ResidualFeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        self.residual_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.output_layer = nn.Linear(128, 256)

    def forward(self, images):

        # Convert RGB to grayscale
        gray = images.mean(dim=1, keepdim=True)

        # Extract high-frequency residual
        blurred = nn.functional.avg_pool2d(
            gray,
            kernel_size=5,
            stride=1,
            padding=2
        )

        residual = gray - blurred

        # Learn residual features
        features = self.residual_conv(residual)

        features = features.flatten(1)

        features = self.output_layer(features)

        return features


residual_extractor = ResidualFeatureExtractor().to(DEVICE)

print("Residual feature extractor initialized!")


Residual feature extractor initialized!


In [44]:

dummy_images = torch.randn(2, 3, 160, 160).to(DEVICE)

residual_features = residual_extractor(dummy_images)

print("Input shape:", dummy_images.shape)
print("Residual feature shape:", residual_features.shape)
print("NaN:", torch.isnan(residual_features).any().item())
print("Inf:", torch.isinf(residual_features).any().item())


Input shape: torch.Size([2, 3, 160, 160])
Residual feature shape: torch.Size([2, 256])
NaN: False
Inf: False


In [45]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class DifferentiableFrequencyExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        # Haar wavelet filters
        s = 1.0 / math.sqrt(2.0)

        haar_filters = torch.tensor([
            [[s * s,  s * s],
             [s * s,  s * s]],

            [[s * s, -s * s],
             [s * s, -s * s]],

            [[s * s,  s * s],
             [-s * s, -s * s]],

            [[s * s, -s * s],
             [-s * s,  s * s]]
        ], dtype=torch.float32).unsqueeze(1)

        self.register_buffer("haar_filters", haar_filters)

    def haar_transform(self, x):

        return F.conv2d(
            x,
            self.haar_filters,
            stride=2
        )

    def forward(self, images):

        # Grayscale conversion
        gray = images.mean(dim=1, keepdim=True)

        # FFT representation
        fft = torch.fft.fft2(gray)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        fft_features = torch.log1p(torch.abs(fft))

        # First Haar decomposition
        level1 = self.haar_transform(gray)

        ll1 = level1[:, 0:1]
        detail1 = level1[:, 1:4]

        # Second Haar decomposition on approximation
        level2 = self.haar_transform(ll1)

        ll2 = level2[:, 0:1]
        detail2 = level2[:, 1:4]

        # Concatenate coefficients
        haar_features = torch.cat([
            ll2.flatten(1),
            detail2.flatten(1),
            detail1.flatten(1)
        ], dim=1)

        return fft_features, haar_features


frequency_extractor = DifferentiableFrequencyExtractor().to(DEVICE)

print("Differentiable frequency extractor initialized!")


Differentiable frequency extractor initialized!


In [46]:

dummy_images = torch.randn(
    2, 3, 160, 160,
    device=DEVICE,
    requires_grad=True
)

fft_features, haar_features = frequency_extractor(dummy_images)

print("FFT shape:", fft_features.shape)
print("Haar shape:", haar_features.shape)

print("FFT NaN:", torch.isnan(fft_features).any().item())
print("Haar NaN:", torch.isnan(haar_features).any().item())

# Verify gradients through Haar extraction
loss = haar_features.mean() + fft_features.mean()
loss.backward()

print("Input gradient exists:", dummy_images.grad is not None)
print("Input gradient NaN:", torch.isnan(dummy_images.grad).any().item())
print("Input gradient non-zero:", dummy_images.grad.abs().sum().item() > 0)


FFT shape: torch.Size([2, 1, 160, 160])
Haar shape: torch.Size([2, 25600])
FFT NaN: False
Haar NaN: False
Input gradient exists: True
Input gradient NaN: False
Input gradient non-zero: True


In [47]:

class FrequencyEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        # FFT feature encoder
        self.fft_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(inplace=True)
        )

        # Haar feature encoder
        self.haar_encoder = nn.Sequential(
            nn.Linear(25600, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True)
        )

        # Combined frequency representation
        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(inplace=True)
        )

    def forward(self, fft_features, haar_features):

        fft_vector = self.fft_encoder(fft_features)
        haar_vector = self.haar_encoder(haar_features)

        combined = torch.cat(
            [fft_vector, haar_vector],
            dim=1
        )

        return self.fusion(combined)


frequency_encoder = FrequencyEncoder().to(DEVICE)

print("Frequency encoder initialized!")


Frequency encoder initialized!


In [48]:

dummy_images = torch.randn(
    2, 3, 160, 160,
    device=DEVICE,
    requires_grad=True
)

fft_features, haar_features = frequency_extractor(dummy_images)

frequency_features = frequency_encoder(
    fft_features,
    haar_features
)

print("FFT shape:", fft_features.shape)
print("Haar shape:", haar_features.shape)
print("Final frequency features:", frequency_features.shape)

print("NaN:", torch.isnan(frequency_features).any().item())
print("Inf:", torch.isinf(frequency_features).any().item())

frequency_features.mean().backward()

print(
    "Gradient flows to input:",
    dummy_images.grad is not None
)


FFT shape: torch.Size([2, 1, 160, 160])
Haar shape: torch.Size([2, 25600])
Final frequency features: torch.Size([2, 256])
NaN: False
Inf: False
Gradient flows to input: True


In [49]:

class AMSFFeatureExtractor(nn.Module):

    def __init__(self, spatial_backbone,
                 frequency_extractor,
                 frequency_encoder,
                 residual_extractor):

        super().__init__()

        self.spatial = spatial_backbone
        self.frequency_extractor = frequency_extractor
        self.frequency_encoder = frequency_encoder
        self.residual = residual_extractor

    def forward(self, images):

        # Spatial branch
        spatial_features = self.spatial(images)

        # Frequency branch
        fft_features, haar_features = (
            self.frequency_extractor(images)
        )

        frequency_features = self.frequency_encoder(
            fft_features,
            haar_features
        )

        # Residual branch
        residual_features = self.residual(images)

        return {
            "spatial": spatial_features,
            "frequency": frequency_features,
            "residual": residual_features
        }


feature_extractor = AMSFFeatureExtractor(
    spatial_backbone,
    frequency_extractor,
    frequency_encoder,
    residual_extractor
).to(DEVICE)

print("AMSF unified feature extractor initialized!")


AMSF unified feature extractor initialized!


In [50]:

feature_extractor.eval()

dummy_images = torch.randn(
    2, 3, 160, 160
).to(DEVICE)

with torch.no_grad():
    features = feature_extractor(dummy_images)

for branch_name, branch_features in features.items():

    print(
        branch_name,
        "shape:",
        branch_features.shape,
        "| NaN:",
        torch.isnan(branch_features).any().item(),
        "| Inf:",
        torch.isinf(branch_features).any().item()
    )


spatial shape: torch.Size([2, 960]) | NaN: False | Inf: False
frequency shape: torch.Size([2, 256]) | NaN: False | Inf: False
residual shape: torch.Size([2, 256]) | NaN: False | Inf: False


In [51]:

feature_extractor.eval()

dummy_images = torch.randn(
    2, 3, 160, 160
).to(DEVICE)

with torch.no_grad():
    features = feature_extractor(dummy_images)

for branch_name, branch_features in features.items():

    print(
        branch_name,
        "shape:",
        branch_features.shape,
        "| NaN:",
        torch.isnan(branch_features).any().item(),
        "| Inf:",
        torch.isinf(branch_features).any().item()
    )


spatial shape: torch.Size([2, 960]) | NaN: False | Inf: False
frequency shape: torch.Size([2, 256]) | NaN: False | Inf: False
residual shape: torch.Size([2, 256]) | NaN: False | Inf: False


# Implement the reliability network

In [52]:

class DegradationConditionedReliability(nn.Module):

    def __init__(self, descriptor_dim=8, hidden_dim=64):
        super().__init__()

        self.reliability_network = nn.Sequential(

            nn.Linear(descriptor_dim, hidden_dim),
            nn.ReLU(inplace=True),

            nn.Linear(hidden_dim, 32),
            nn.ReLU(inplace=True),

            nn.Linear(32, 3),
            nn.Sigmoid()
        )

    def forward(self, degradation_descriptors):

        reliability_scores = self.reliability_network(
            degradation_descriptors
        )

        return reliability_scores


reliability_module = DegradationConditionedReliability().to(DEVICE)

print("Degradation-conditioned reliability module initialized!")


Degradation-conditioned reliability module initialized!


In [53]:

# Simulate a batch of 2 images with 8 normalized descriptors
dummy_descriptors = torch.randn(
    2, 8,
    device=DEVICE
)

reliability_scores = reliability_module(dummy_descriptors)

print("Descriptor input shape:", dummy_descriptors.shape)
print("Reliability output shape:", reliability_scores.shape)

print("\nReliability scores:")
print(reliability_scores.detach().cpu().numpy())

print("\nAll scores between 0 and 1:",
      ((reliability_scores >= 0) &
       (reliability_scores <= 1)).all().item())

print("NaN:", torch.isnan(reliability_scores).any().item())
print("Inf:", torch.isinf(reliability_scores).any().item())

Descriptor input shape: torch.Size([2, 8])
Reliability output shape: torch.Size([2, 3])

Reliability scores:
[[0.5476274  0.43541068 0.505862  ]
 [0.5257778  0.44268543 0.49023023]]

All scores between 0 and 1: True
NaN: False
Inf: False


In [54]:

class ReliabilityWeightedFusion(nn.Module):

    def __init__(self, feature_dim=256):
        super().__init__()

        # Project spatial features: 960 -> 256
        self.spatial_projection = nn.Sequential(
            nn.Linear(960, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU(inplace=True)
        )

        # Frequency features: 256 -> 256
        self.frequency_projection = nn.Sequential(
            nn.Linear(256, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU(inplace=True)
        )

        # Residual features: 256 -> 256
        self.residual_projection = nn.Sequential(
            nn.Linear(256, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, features, reliability_scores):

        spatial = self.spatial_projection(
            features["spatial"]
        )

        frequency = self.frequency_projection(
            features["frequency"]
        )

        residual = self.residual_projection(
            features["residual"]
        )

        # Create three feature nodes
        nodes = torch.stack(
            [spatial, frequency, residual],
            dim=1
        )

        # Apply branch-specific reliability scores
        weighted_nodes = (
            nodes * reliability_scores.unsqueeze(-1)
        )

        return weighted_nodes


fusion_module = ReliabilityWeightedFusion().to(DEVICE)

print("Reliability-weighted fusion module initialized!")


Reliability-weighted fusion module initialized!


In [55]:

# Generate test images
dummy_images = torch.randn(
    2, 3, 160, 160,
    device=DEVICE
)

# Extract branch features
feature_extractor.eval()

with torch.no_grad():
    extracted_features = feature_extractor(dummy_images)

# Generate degradation descriptors
dummy_descriptors = torch.randn(
    2, 8,
    device=DEVICE
)

# Estimate reliability
reliability_scores = reliability_module(
    dummy_descriptors
)

# Apply reliability-weighted fusion
weighted_nodes = fusion_module(
    extracted_features,
    reliability_scores
)

print("Spatial features:", extracted_features["spatial"].shape)
print("Frequency features:", extracted_features["frequency"].shape)
print("Residual features:", extracted_features["residual"].shape)

print("\nReliability scores:", reliability_scores.shape)
print("Weighted graph nodes:", weighted_nodes.shape)

print("NaN:", torch.isnan(weighted_nodes).any().item())
print("Inf:", torch.isinf(weighted_nodes).any().item())


Spatial features: torch.Size([2, 960])
Frequency features: torch.Size([2, 256])
Residual features: torch.Size([2, 256])

Reliability scores: torch.Size([2, 3])
Weighted graph nodes: torch.Size([2, 3, 256])
NaN: False
Inf: False


In [56]:

class GraphAttentionLayer(nn.Module):

    def __init__(self, feature_dim=256):
        super().__init__()

        self.query = nn.Linear(feature_dim, feature_dim)
        self.key = nn.Linear(feature_dim, feature_dim)
        self.value = nn.Linear(feature_dim, feature_dim)

        self.scale = feature_dim ** -0.5

        self.output_projection = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, nodes):

        Q = self.query(nodes)
        K = self.key(nodes)
        V = self.value(nodes)

        # Calculate node-to-node attention
        attention_scores = torch.matmul(
            Q, K.transpose(-2, -1)
        ) * self.scale

        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        # Aggregate information from connected nodes
        aggregated = torch.matmul(
            attention_weights, V
        )

        output = self.output_projection(aggregated)

        # Residual connection
        output = output + nodes

        return output, attention_weights


graph_attention = GraphAttentionLayer().to(DEVICE)

print("Graph attention layer initialized!")


Graph attention layer initialized!


In [57]:

dummy_nodes = torch.randn(
    2, 3, 256,
    device=DEVICE
)

graph_output, attention_weights = graph_attention(
    dummy_nodes
)

print("Input graph shape:", dummy_nodes.shape)
print("Graph output shape:", graph_output.shape)
print("Attention weights shape:", attention_weights.shape)

print("\nAttention weights:")
print(attention_weights.detach().cpu().numpy())

print("\nAttention sums:")
print(attention_weights.sum(dim=-1).detach().cpu().numpy())

print("\nNaN:", torch.isnan(graph_output).any().item())
print("Inf:", torch.isinf(graph_output).any().item())


Input graph shape: torch.Size([2, 3, 256])
Graph output shape: torch.Size([2, 3, 256])
Attention weights shape: torch.Size([2, 3, 3])

Attention weights:
[[[0.2554409  0.36767963 0.37687942]
  [0.1094076  0.47707608 0.41351634]
  [0.35425732 0.425367   0.2203757 ]]

 [[0.36541927 0.25197738 0.3826033 ]
  [0.21288806 0.30340606 0.48370582]
  [0.49145412 0.2975739  0.21097198]]]

Attention sums:
[[0.9999999  1.         1.        ]
 [0.99999994 0.99999994 1.        ]]

NaN: False
Inf: False


In [58]:

class MultiHeadGraphAttention(nn.Module):

    def __init__(self, feature_dim=256, num_heads=4):
        super().__init__()

        assert feature_dim % num_heads == 0

        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads

        self.query = nn.Linear(feature_dim, feature_dim)
        self.key = nn.Linear(feature_dim, feature_dim)
        self.value = nn.Linear(feature_dim, feature_dim)

        self.output_projection = nn.Linear(
            feature_dim, feature_dim
        )

        self.norm = nn.LayerNorm(feature_dim)

        self.scale = self.head_dim ** -0.5

    def forward(self, nodes):

        batch_size, num_nodes, _ = nodes.shape

        Q = self.query(nodes)
        K = self.key(nodes)
        V = self.value(nodes)

        # Separate attention heads
        Q = Q.view(
            batch_size, num_nodes,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size, num_nodes,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size, num_nodes,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        # Attention calculation
        scores = torch.matmul(
            Q, K.transpose(-2, -1)
        ) * self.scale

        attention = torch.softmax(scores, dim=-1)

        attended = torch.matmul(attention, V)

        # Combine attention heads
        attended = attended.transpose(1, 2).contiguous()

        attended = attended.view(
            batch_size, num_nodes, self.feature_dim
        )

        output = self.output_projection(attended)

        output = self.norm(output + nodes)

        return output, attention


multihead_graph = MultiHeadGraphAttention(
    feature_dim=256,
    num_heads=4
).to(DEVICE)

print("Multi-head graph attention initialized!")


Multi-head graph attention initialized!


In [59]:

dummy_nodes = torch.randn(
    2, 3, 256,
    device=DEVICE
)

graph_output, attention_weights = multihead_graph(
    dummy_nodes
)

print("Input:", dummy_nodes.shape)
print("Output:", graph_output.shape)
print("Attention:", attention_weights.shape)

print(
    "Attention row sums:",
    attention_weights.sum(dim=-1)
)

print("NaN:", torch.isnan(graph_output).any().item())
print("Inf:", torch.isinf(graph_output).any().item())


Input: torch.Size([2, 3, 256])
Output: torch.Size([2, 3, 256])
Attention: torch.Size([2, 4, 3, 3])
Attention row sums: tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]], device='cuda:0', grad_fn=<SumBackward1>)
NaN: False
Inf: False


In [60]:

class AdaptiveBranchGating(nn.Module):

    def __init__(self, feature_dim=256, hidden_dim=128):
        super().__init__()

        self.gate_network = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),

            nn.Linear(hidden_dim, 64),
            nn.ReLU(inplace=True),

            nn.Linear(64, 1)
        )

    def forward(self, graph_nodes):

        # graph_nodes: (B, 3, 256)

        # Calculate importance score for each branch
        gate_logits = self.gate_network(
            graph_nodes
        ).squeeze(-1)

        # Normalize branch importance
        gate_weights = torch.softmax(
            gate_logits, dim=1
        )

        # Weighted feature fusion
        fused_features = torch.sum(
            graph_nodes * gate_weights.unsqueeze(-1),
            dim=1
        )

        return fused_features, gate_weights


adaptive_gating = AdaptiveBranchGating(
    feature_dim=256,
    hidden_dim=128
).to(DEVICE)

print("Adaptive Branch Gating initialized successfully!")


Adaptive Branch Gating initialized successfully!


In [61]:

# Generate dummy graph nodes
dummy_nodes = torch.randn(
    2, 3, 256,
    device=DEVICE
)

# Apply multi-head graph attention
graph_output, attention_weights = multihead_graph(
    dummy_nodes
)

# Apply adaptive branch gating
fused_features, gate_weights = adaptive_gating(
    graph_output
)

print("Graph output shape:", graph_output.shape)

print("Fused feature shape:", fused_features.shape)

print("Gate weights shape:", gate_weights.shape)

print("\nBranch importance weights:")
print(gate_weights.detach().cpu().numpy())

print("\nGate weight sums:")
print(gate_weights.sum(dim=1).detach().cpu().numpy())

print("\nNaN:", torch.isnan(fused_features).any().item())
print("Inf:", torch.isinf(fused_features).any().item())


Graph output shape: torch.Size([2, 3, 256])
Fused feature shape: torch.Size([2, 256])
Gate weights shape: torch.Size([2, 3])

Branch importance weights:
[[0.310431   0.33597454 0.35359448]
 [0.34589872 0.32807055 0.32603073]]

Gate weight sums:
[1. 1.]

NaN: False
Inf: False


In [62]:

class AMSFClassifier(nn.Module):

    def __init__(self, input_dim=256, num_classes=2):
        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(64, num_classes)
        )

    def forward(self, fused_features):
        logits = self.classifier(fused_features)
        return logits


classifier = AMSFClassifier(
    input_dim=256,
    num_classes=2
).to(DEVICE)

print("AMSF classifier initialized successfully!")


AMSF classifier initialized successfully!


In [63]:

dummy_features = torch.randn(
    2, 256,
    device=DEVICE
)

logits = classifier(dummy_features)

probabilities = torch.softmax(logits, dim=1)

predictions = torch.argmax(probabilities, dim=1)

print("Input features:", dummy_features.shape)
print("Logits:", logits.shape)
print("Probabilities:", probabilities.detach().cpu().numpy())
print("Predictions:", predictions.detach().cpu().numpy())

print("NaN:", torch.isnan(logits).any().item())
print("Inf:", torch.isinf(logits).any().item())


Input features: torch.Size([2, 256])
Logits: torch.Size([2, 2])
Probabilities: [[0.5216209  0.47837907]
 [0.28341112 0.7165889 ]]
Predictions: [0 1]
NaN: False
Inf: False


In [64]:

required_objects = [
    "spatial_backbone",
    "frequency_extractor",
    "frequency_encoder",
    "residual_extractor",
    "adaptive_gating",
    "classifier",
    "DEVICE"
]

for name in required_objects:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")


spatial_backbone: Available
frequency_extractor: Available
frequency_encoder: Available
residual_extractor: Available
adaptive_gating: Available
classifier: Available
DEVICE: Available


# Define the complete model

In [65]:

import copy
import torch
import torch.nn as nn

class AMSFGNetV2(nn.Module):

    def __init__(
        self,
        spatial_backbone,
        frequency_extractor,
        frequency_encoder,
        residual_extractor,
        use_reliability=True
    ):
        super().__init__()

        self.use_reliability = use_reliability

        # Feature extraction branches
        self.spatial = spatial_backbone
        self.frequency_extractor = frequency_extractor
        self.frequency_encoder = frequency_encoder
        self.residual = residual_extractor

        # ImageNet normalization for spatial branch only
        self.register_buffer(
            "imagenet_mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "imagenet_std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

        # Degradation-conditioned reliability
        self.reliability = DegradationConditionedReliability(
            descriptor_dim=8,
            hidden_dim=64
        )

        # Project and combine the three branch features
        self.reliability_fusion = ReliabilityWeightedFusion(
            feature_dim=256
        )

        # Multi-head graph attention
        self.graph_attention = MultiHeadGraphAttention(
            feature_dim=256,
            num_heads=4
        )

        # Adaptive branch gating
        self.adaptive_gating = AdaptiveBranchGating(
            feature_dim=256,
            hidden_dim=128
        )

        # Final classifier
        self.classifier = AMSFClassifier(
            input_dim=256,
            num_classes=2
        )

    def forward(self, raw_images, degradation_descriptors):

        # raw_images must be RGB in [0, 1]
        # degradation_descriptors must have 8 normalized values

        # 1. Spatial branch: ImageNet-normalized input
        normalized_images = (
            raw_images - self.imagenet_mean
        ) / self.imagenet_std

        spatial_features = self.spatial(normalized_images)

        # 2. Frequency branch: raw image input
        fft_features, haar_features = self.frequency_extractor(
            raw_images
        )

        frequency_features = self.frequency_encoder(
            fft_features,
            haar_features
        )

        # 3. Residual branch: raw image input
        residual_features = self.residual(raw_images)

        branch_features = {
            "spatial": spatial_features,
            "frequency": frequency_features,
            "residual": residual_features
        }

        # 4. Reliability scores
        if self.use_reliability:
            reliability_scores = self.reliability(
                degradation_descriptors
            )
        else:
            # Ablation: no learned reliability weighting
            reliability_scores = torch.ones(
                raw_images.size(0),
                3,
                device=raw_images.device,
                dtype=raw_images.dtype
            )

        # 5. Reliability-weighted feature fusion
        nodes = self.reliability_fusion(
            branch_features,
            reliability_scores
        )

        # 6. Multi-head graph attention
        graph_nodes, graph_attention_weights = (
            self.graph_attention(nodes)
        )

        # 7. Adaptive branch gating
        fused_features, gate_weights = self.adaptive_gating(
            graph_nodes
        )

        # 8. REAL / FAKE classification
        logits = self.classifier(fused_features)

        return {
            "logits": logits,
            "reliability_scores": reliability_scores,
            "graph_attention": graph_attention_weights,
            "gate_weights": gate_weights,
            "fused_features": fused_features
        }


# Create Model A: Full AMSF-GNet V2
model_A = AMSFGNetV2(
    spatial_backbone=copy.deepcopy(spatial_backbone),
    frequency_extractor=copy.deepcopy(frequency_extractor),
    frequency_encoder=copy.deepcopy(frequency_encoder),
    residual_extractor=copy.deepcopy(residual_extractor),
    use_reliability=True
).to(DEVICE)


# Create Model B: Ablation without learned reliability
model_B = AMSFGNetV2(
    spatial_backbone=copy.deepcopy(spatial_backbone),
    frequency_extractor=copy.deepcopy(frequency_extractor),
    frequency_encoder=copy.deepcopy(frequency_encoder),
    residual_extractor=copy.deepcopy(residual_extractor),
    use_reliability=False
).to(DEVICE)


print("Model A initialized: Full AMSF-GNet V2")
print("Model B initialized: Reliability ablation")
print("Device:", DEVICE)


Model A initialized: Full AMSF-GNet V2
Model B initialized: Reliability ablation
Device: cuda


In [66]:

# Create dummy input images in [0, 1]
dummy_images = torch.rand(
    2, 3, 160, 160,
    device=DEVICE
)

# Eight normalized degradation descriptors
dummy_descriptors = torch.randn(
    2, 8,
    device=DEVICE
)

# Test Model A
model_A.eval()

with torch.no_grad():
    output_A = model_A(
        dummy_images,
        dummy_descriptors
    )

print("========== MODEL A ==========")
print("Logits:", output_A["logits"].shape)
print("Reliability:", output_A["reliability_scores"].shape)
print("Graph attention:", output_A["graph_attention"].shape)
print("Gate weights:", output_A["gate_weights"].shape)
print("Fused features:", output_A["fused_features"].shape)


# Test Model B
model_B.eval()

with torch.no_grad():
    output_B = model_B(
        dummy_images,
        dummy_descriptors
    )

print("\n========== MODEL B ==========")
print("Logits:", output_B["logits"].shape)
print("Reliability:", output_B["reliability_scores"].shape)
print("Graph attention:", output_B["graph_attention"].shape)
print("Gate weights:", output_B["gate_weights"].shape)
print("Fused features:", output_B["fused_features"].shape)


# Numerical checks
print("\n========== VALIDATION ==========")

for name, output in [
    ("Model A", output_A),
    ("Model B", output_B)
]:
    print(f"\n{name}")
    print("NaN:", torch.isnan(output["logits"]).any().item())
    print("Inf:", torch.isinf(output["logits"]).any().item())
    print(
        "Gate weights sum:",
        output["gate_weights"].sum(dim=1).cpu().numpy()
    )


========== MODEL A ==========
Logits: torch.Size([2, 2])
Reliability: torch.Size([2, 3])
Graph attention: torch.Size([2, 4, 3, 3])
Gate weights: torch.Size([2, 3])
Fused features: torch.Size([2, 256])

========== MODEL B ==========
Logits: torch.Size([2, 2])
Reliability: torch.Size([2, 3])
Graph attention: torch.Size([2, 4, 3, 3])
Gate weights: torch.Size([2, 3])
Fused features: torch.Size([2, 256])

========== VALIDATION ==========

Model A
NaN: False
Inf: False
Gate weights sum: [0.99999994 1.        ]

Model B
NaN: False
Inf: False
Gate weights sum: [1. 1.]


In [67]:

import torch
import torch.nn as nn

# ==========================================
# 1. CLEAR PREVIOUS GRADIENTS AND GPU CACHE
# ==========================================

model_A.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(DEVICE)

# ==========================================
# 2. SET MODEL TO TRAINING MODE
# ==========================================

model_A.train()

# ==========================================
# 3. CREATE DUMMY TRAINING BATCH
# ==========================================

test_images = torch.rand(
    2, 3, 160, 160,
    device=DEVICE
)

test_descriptors = torch.randn(
    2, 8,
    device=DEVICE
)

test_labels = torch.tensor(
    [0, 1],
    dtype=torch.long,
    device=DEVICE
)

# ==========================================
# 4. FORWARD PASS
# ==========================================

output = model_A(
    test_images,
    test_descriptors
)

# ==========================================
# 5. CALCULATE CLASSIFICATION LOSS
# ==========================================

criterion = nn.CrossEntropyLoss()

loss = criterion(
    output["logits"],
    test_labels
)

# ==========================================
# 6. BACKWARD PASS
# ==========================================

loss.backward()

# ==========================================
# 7. GRADIENT FLOW VERIFICATION
# ==========================================

gradient_checks = {
    "Spatial branch":
        model_A.spatial.features[0][0].weight.grad,

    "Frequency encoder":
        model_A.frequency_encoder.fft_encoder[0].weight.grad,

    "Residual branch":
        model_A.residual.output_layer.weight.grad,

    "Reliability network":
        model_A.reliability.reliability_network[0].weight.grad,

    "Graph attention":
        model_A.graph_attention.query.weight.grad,

    "Adaptive gating":
        model_A.adaptive_gating.gate_network[0].weight.grad,

    "Classifier":
        model_A.classifier.classifier[0].weight.grad
}

print("=" * 50)
print("GRADIENT FLOW VERIFICATION")
print("=" * 50)

all_gradients_passed = True

for name, grad in gradient_checks.items():

    passed = (
        grad is not None
        and torch.isfinite(grad).all().item()
        and grad.abs().sum().item() > 0
    )

    if not passed:
        all_gradients_passed = False

    print(f"{name}: {'PASS' if passed else 'FAIL'}")

# ==========================================
# 8. LOSS VERIFICATION
# ==========================================

print("\nLOSS VERIFICATION")

print("Training loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

# ==========================================
# 9. GPU MEMORY VERIFICATION
# ==========================================

print("\nGPU MEMORY VERIFICATION")

if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated(DEVICE) / (1024 ** 3)
    )

    peak = (
        torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved(DEVICE) / (1024 ** 3)
    )

    print(f"Current allocated: {allocated:.2f} GB")
    print(f"Peak allocated: {peak:.2f} GB")
    print(f"Reserved memory: {reserved:.2f} GB")

# ==========================================
# 10. FINAL VERIFICATION
# ==========================================

print("\n" + "=" * 50)

print(
    "GRADIENT VERIFICATION:",
    "PASSED" if all_gradients_passed else "FAILED"
)

print(
    "LOSS VERIFICATION:",
    "PASSED" if torch.isfinite(loss).item() else "FAILED"
)

print("=" * 50)

# Clear test gradients and restore evaluation mode
model_A.zero_grad(set_to_none=True)
model_A.eval()


GRADIENT FLOW VERIFICATION
Spatial branch: PASS
Frequency encoder: PASS
Residual branch: PASS
Reliability network: PASS
Graph attention: PASS
Adaptive gating: PASS
Classifier: PASS

LOSS VERIFICATION
Training loss: 0.683774471282959
Loss finite: True

GPU MEMORY VERIFICATION
Current allocated: 0.45 GB
Peak allocated: 0.52 GB
Reserved memory: 0.63 GB

GRADIENT VERIFICATION: PASSED
LOSS VERIFICATION: PASSED


AMSFGNetV2(
  (spatial): MobileNetV3(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
          )
        )
      )
      (2): InvertedResidual(
        (block): Sequential(
  